# Exploring displacement threshold energy with molecular dynamics

In this notebook we will run a series of short simulations in a small system to explore the displacement threshold energy for a model of body-centred cubic iron.

## Code to set-up Google Colab environment

### Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Clone repository of exercises

In [ ]:
%%bash
cd /content/drive/MyDrive/
mkdir -p AdvancedNuclearSystems
cd AdvancedNuclearSystems
git clone https://github.com/AtomisticSimulationOfMaterials/DisplacementThresholdEnergy.git

### Install the lammps molecular dynamics software


In [ ]:
# Install lammps
!pip install lammps[mpi]
# Install package to assist in parsing lammps log files
!pip install lammps-logfile

### Load some modules 

In [ ]:
import numpy as np
from scipy import optimize
import os
import shutil
import subprocess
from contextlib import chdir
import lammps_logfile
import matplotlib.pyplot as plt
import random
%matplotlib inline

### Change to the correct directory

In [ ]:
%cd /content/drive/MyDrive/AdvancedNuclearSystems/DisplacementThresholdEnergy/01-DisplacementThreshold

### Define some useful functions and constants


In [ ]:
# Define some physical constants we will need
m_Fe = 55.845 # amu
eV = 1.602e-19 # J
amu = 1.66e-27 # kg

# Define a function to replace text in base input file for lammps
def replace_all(text, dic):
    for i, j in dic.items():
        text = text.replace(i, j)
    return text

# Define a function to convert a kinetic energy in eV to a speed in Ang/ps
def v_from_E(ke, dirn, mass):
    # Return velocity vector in units of A ps-1
    return np.sqrt(2.0*ke*eV/(mass*amu))*1e-2*dirn/np.linalg.norm(dirn)

# Define a function to calculate the number of defective atoms at the end of the simulation
def count_defects(path, step):
    data = np.loadtxt(path + 'dump.' + str(step) + '.txt', skiprows=9)
    return (data[:,11] > -3.7).sum()

## Run simulations to explore displacement threshold energy

### Define a function to set up and run a simulation

The function below does a lot, so it is worth spending some time to understand the contents. The function does the following (see also the comments in the function code).
1. Create a folder for the simulation and copy the file that contains details of the empirical potential to the folder.
2. Select a random seed to ensure that each simulation is slightly different.
3. Calculate the speed corresponding to the desired kinetic energy.
4. Define the values to use in this simulation. These will replace placeholders in the base input file.
5. Take the base file of instructions (`InputFile/displacement_base.lmps`) for the simulations and create a copy in the simulation folder, making substitutions for the placeholder values .
6. Define the command required to run lammps.
7. Run the simulation.


In [ ]:
def create_and_run(ke, dirn, temp, index):
    # 1. Create a folder for the simulation and copy the file that contains details of the empirical potential to the folder.
    path = '../Simulations/' + 'dirn_' + str(dirn[0]) + '_' + str(dirn[1]) + '_' + str(dirn[2]) + '/' + 'T_' + str(T) + '/' + 'ke_' + str(ke) +  '/' + 'run_' + str(index) + '/'
    if not os.path.exists(path):
        os.makedirs(path, mode=0o777)
    shutil.copy2('../Potentials/' + potential_file, path)

    # 2. Select a random seed to ensure that each simulation is slightly different
    seed = random.randint(1,30000)
    
    # 3. Calculate the speed corresponding to the desired kinetic energy.
    vel = v_from_E(ke,np.array(dirn), m_Fe)
    
    # 4. Define the values to use in this simulation. These will replace placeholders in the base input file.
    replacements = {
        '<pkaid>':str(pkaid),
        '<seed>':str(seed),
        '<temp>':str(temp),
        '<vx>':str(vel[0]),
        '<vy>':str(vel[1]),
        '<vz>':str(vel[2]),
        '<thermosteps>':str(thermosteps),
        '<runsteps>':str(runsteps),
        '<fdump>':str(fdump)
        }
            
    # 5. Take the base file of instructions (InputFile/displacement_base.lmp`) for the simulations 
    #        and create a copy in the simulation folder, making substitutions for the placeholder values 
    inputLammpsFile = '../InputFile/displacement_base.lmps'
    outputLammpsFile = path + 'in.lmps'        
    finLammps = open(inputLammpsFile, 'r').read()
    foutLammps = open(outputLammpsFile, 'w')
    out = replace_all(finLammps, replacements)
    foutLammps.write(out)
    foutLammps.close()


    # 6. Define the command required to run lammps
    lammps_executable = 'lmp'
    lammps_command = '-in in.lmps'

    # 7. Run the simulation
    with chdir(path):
        os.system(lammps_executable + ' ' + lammps_command + ' >/dev/null 2>&1')

    return path

### Define some of the simulation parameters common to all the simulations

In [ ]:
potential_file = 'Fe_2.eam.fs'
pkaid = 126
thermosteps = 2000
runsteps = 10000
fdump = 1000

### Run a series of simulations to explore the displacement threshold energy

You will need to insert your chosen direction (selected from the class google sheet)

In [ ]:
# Define your chosen direction here, by replacing the values of h,k,l
direction = [1,2,3] #[h,k,l]
# Specify the temperature
T = 300
# Set the number of equivalent simulations to run (in order to gather statistics)
n_runs = 2

# Define a list of values of the kinetic energy (in eV) to explore
kinetic_energies = [20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 85, 90, 95, 100, 105, 110, 115, 120, 125, 130, 135, 140, 145, 150]

# Set up an empty list to hold the calculated thresholds
thresholds = []

# Set up a loop to run the simulations multiple times to gather statistics
for i_run in range(n_runs):
    # Initialise some values to record the threshold for this run
    i_ke = 0
    thresh_found = False
    # Loop over the kinetic energies in the list
    while not thresh_found and i_ke < len(kinetic_energies):
        # Run a simulation at the current energy
        path = create_and_run(kinetic_energies[i_ke], direction, T, i_run)
        # Check to see if a defect has been created, and if so record the current KE and abort the loop over energies.
        # If the highrest KE in the list is reached without creating a defect, record a value of 999 as an indicator.
        if int(count_defects(path, runsteps + 2*thermosteps)) > 0:
            thresh_found = True
            thresholds.append(kinetic_energies[i_ke])
        elif i_ke == len(kinetic_energies)-1:
            thresholds.append(999)
        i_ke = i_ke + 1

# Write out the results to a file
np.savetxt('../Simulations/'  + 'dirn_' + str(direction[0]) + '_' + str(direction[1]) + '_' + str(direction[2]) + '/' + 'T_' + str(T) + '/'+ 'thresholds.txt', thresholds)

### Calculate key statistics for the simulations

Please also add your results to the google sheet

In [ ]:
print('Statistics for displacement threshold energy at T=' + str(T) +'K, for direction [' + str(direction[0]) + ' ' + str(direction[1]) + ' ' + str(direction[2]) + ']')
print('-----------------------------------------------------------------------------')
print('Mean      : ' + str(np.mean(thresholds)))
print('Std Error : ' + str(np.std(thresholds)/np.sqrt(n_runs - 1)))
print('Std Dev   : ' + str(np.std(thresholds)))
print('Max       : ' + str(np.max(thresholds)))
print('Min       : ' + str(np.min(thresholds)))